## 8. Scale to full 56K

Once the prototype looks clean, run `compute_legnet_attrs.py` via the sbatch script in this dir to produce per-seq attributions for the whole library; this notebook can then load those and recompute axes.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(proto['dlog2FC'], proto['cos_HK'],
                c=proto['HepG2_log2FC'] + proto['K562_log2FC'],  # joint activity
                cmap='inferno', s=14, alpha=0.7)
ax.axhline(0, color='k', lw=0.4); ax.axvline(0, color='k', lw=0.4)
ax.set_xlabel('measured  HepG2 log2FC − K562 log2FC')
ax.set_ylabel('attribution cos sim (HepG2 vs K562)')
ax.set_title(f'k-cee on MPRA-LegNet ({len(proto)} seqs, prototype)')
plt.colorbar(sc, label='HepG2 + K562 log2FC (joint activity)')
plt.tight_layout(); plt.show()

## 7. k-cee scatter — func (x) vs mech (y)

Func = measured `HepG2_log2FC − K562_log2FC`; mech = cosine similarity of attributions. Low cos sim → divergent mechanism between cell lines.

In [ ]:
X_np = X.numpy()  # (N, 4, L)
obs_H = (attr_H * X_np).sum(axis=1)  # (N, L)
obs_K = (attr_K * X_np).sum(axis=1)

def cos_sim(a, b, eps=1e-12):
    return (a * b).sum(-1) / (np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1) + eps)

cos_HK = cos_sim(obs_H, obs_K)
proto['cos_HK'] = cos_HK
print(f"cos sim H/K — mean={cos_HK.mean():.3f}, range [{cos_HK.min():.3f}, {cos_HK.max():.3f}]")

## 6. Mech axis — cosine similarity of H vs K attributions

Project hypothetical → observed (multiply by one-hot, sum channels) per seq, then cosine similarity across the L=230 base positions.

In [ ]:
class _Wrap(torch.nn.Module):
    """Add a trailing dim so tangermeme can index target=0."""
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return self.m(x).unsqueeze(-1)

N_SHUFFLES = 5  # 5 for prototype; 20 for the full 56K run

# Pre-compute dinuc shuffles once (shared across all 20 forward passes)
print("computing dinuc-shuffle references...")
refs = dinucleotide_shuffle(X, n=N_SHUFFLES, random_state=42)  # (N, n_shuf, 4, L)

def ensemble_attr(models, X, refs):
    L = X.shape[-1]
    acc = np.zeros((len(X), 4, L), dtype=np.float32)
    for i, m in enumerate(models):
        a = deep_lift_shap(_Wrap(m).eval(), X, references=refs, target=0,
                           hypothetical=True, batch_size=256, device=DEVICE, verbose=False)
        acc += a.cpu().numpy()
        print(f"  fold {i+1}/{len(models)} done")
    return acc / len(models)

print("HepG2 attributions...")
attr_H = ensemble_attr(models_H, X, refs)
print("K562 attributions...")
attr_K = ensemble_attr(models_K, X, refs)
print(f"attr shapes: H={attr_H.shape}, K={attr_K.shape}")

## 5. DeepLIFT-SHAP attributions (per cell line, ensemble-averaged)

`deep_lift_shap` with `hypothetical=True` returns shape `(N, 4, L)` — attribution at every base, even unobserved ones. We project onto the observed alphabet (`obs = (attr * X).sum(axis=1)` → `(N, L)`) before cosine similarity.

In [ ]:
@torch.no_grad()
def ensemble_predict(models, X, batch_size=512):
    # Returns (N,) numpy of mean prediction across folds.
    out = []
    for m in models:
        ys = []
        for i in range(0, len(X), batch_size):
            ys.append(m(X[i:i+batch_size].to(DEVICE)).cpu().numpy().flatten())
        out.append(np.concatenate(ys))
    return np.mean(out, axis=0).astype(np.float32)

pred_H = ensemble_predict(models_H, X)
pred_K = ensemble_predict(models_K, X)
proto['pred_HepG2'] = pred_H
proto['pred_K562']  = pred_K
proto['pred_dlog2FC'] = pred_H - pred_K

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(proto['HepG2_log2FC'], pred_H, s=8, alpha=0.5); axes[0].set(title='HepG2: meas vs pred', xlabel='measured log2FC', ylabel='legnet pred')
axes[1].scatter(proto['K562_log2FC'], pred_K, s=8, alpha=0.5); axes[1].set(title='K562: meas vs pred',  xlabel='measured log2FC', ylabel='legnet pred')
for a in axes: a.axline((0,0), slope=1, color='k', lw=0.5, ls='--')
plt.tight_layout(); plt.show()

## 4. Predictions (sanity check)

In [ ]:
def load_ensemble(ct, n_folds=10):
    d = LEGNET / 'weights' / ct / 'md_shift_reverse_noavg_noch'
    cfg = TrainingConfig.from_json(str(d / 'config.json'))
    models = []
    for fold in range(1, n_folds + 1):
        cps = sorted(d.glob(f'best_model_test{fold}_val*.ckpt'))
        if not cps:
            print(f"  {ct} fold {fold}: no ckpt")
            continue
        m = LitModel.load_from_checkpoint(str(cps[0]), tr_cfg=cfg)
        m.eval(); m.model.to(DEVICE)
        models.append(m.model)
    print(f"{ct}: loaded {len(models)} fold models on {DEVICE}")
    return models

models_H = load_ensemble('HepG2')
models_K = load_ensemble('K562')

## 3. Load LegNet ensembles

10-fold ensemble per cell line. Picks the first matching `val*` ckpt for each test fold (same convention as the wsbs `compute_legnet_attrs.py`).

In [ ]:
to_tensor = Seq2Tensor()
X = torch.stack([to_tensor(s) for s in proto['sequence']]).float()  # (N, 4, 230)
print(f"X shape: {tuple(X.shape)}, dtype={X.dtype}")
assert X.shape[1] == 4 and X.shape[2] == 230

## 2. One-hot encode (AGCT, channels-first)

LegNet alphabet: A=0, G=1, C=2, T=3 (see `human_legnet/utils.py::CODES`). Output shape `(N, 4, 230)`.

In [ ]:
# Prototype on a stratified 200-seq subset spanning the log2FC diff range
df['dlog2FC'] = df['HepG2_log2FC'] - df['K562_log2FC']
N_PROTO = 200
proto = df.sample(n=N_PROTO, random_state=0).reset_index(drop=True)
print(f"prototype subset: {len(proto)} seqs   dlog2FC range [{proto.dlog2FC.min():.2f}, {proto.dlog2FC.max():.2f}]")

In [ ]:
df = pd.read_csv(REPO / 'data' / 'joint_library_combined.csv')
df = df.dropna(subset=['sequence', 'HepG2_log2FC', 'K562_log2FC']).reset_index(drop=True)
print(f"library: {len(df)} seqs, length={df['sequence'].str.len().iloc[0]} bp")
df.head(2)

## 1. Load library

In [ ]:
import os, sys, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

REPO = Path('/grid/koo/home/pmantill/projects/Virtual_Experiments/Hippo_axis/Hippo_dependency_mpra')
LEGNET = REPO / 'legnet_rep'
sys.path.insert(0, str(LEGNET / 'human_legnet'))

from trainer import LitModel, TrainingConfig
from utils import Seq2Tensor
from tangermeme.deep_lift_shap import deep_lift_shap
from tangermeme.ersatz import dinucleotide_shuffle

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {DEVICE}")

# k-cee on MPRA-LegNet: H vs K attribution cosine similarity

Repeat the early AlphaGenome 2D-targeting analysis with MPRA-LegNet on the 56K joint library.

**Pipeline**
1. Load 56K seqs + log2FC from `data/joint_library_combined.csv`
2. Load LegNet 10-fold ensembles for HepG2 and K562
3. DeepLIFT-SHAP attributions (tangermeme, dinuc-shuffle ref) per cell line
4. **Mech axis** = cosine similarity of observed H vs K attributions per seq
5. **Func axis** = `HepG2_log2FC − K562_log2FC`
6. Scatter (func on x, mech on y)

This notebook prototypes on a 200-seq subset; full 56K runs via `compute_legnet_attrs.py` + sbatch.